# Simple Forecasting Methods

Before reaching for ARIMA, exponential smoothing, or neural networks, every
forecaster should know the **four simple benchmark methods**.  These are
trivially easy to implement, require no parameter tuning, and serve a critical
purpose: **if your fancy model cannot beat these, it is not worth using.**

The four methods are:

| Method | Forecast | Idea |
|---|---|---|
| Mean | $\hat{y}_{T+h|T} = \bar{y}$ | Future = historical average |
| Naive | $\hat{y}_{T+h|T} = y_T$ | Future = last observation |
| Seasonal Naive | $\hat{y}_{T+h|T} = y_{T+h-m(k+1)}$ | Future = same season last year |
| Drift | $\hat{y}_{T+h|T} = y_T + h\left(\frac{y_T - y_1}{T-1}\right)$ | Extrapolate line from first to last |

This notebook implements each from scratch, applies all four to airline
passengers data, and compares their forecasts visually.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# Load airline passengers data
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

print(f"Shape: {airline.shape}")
print(f"Date range: {airline.index[0].date()} to {airline.index[-1].date()}")
airline.head()

## Train / Test Split

For time series we **always** split by time — never randomly.  We will use
the first 120 months (10 years) for training and the last 24 months (2 years)
as the test set.

In [ ]:
TRAIN_SIZE = 120
HORIZON = len(airline) - TRAIN_SIZE  # 24 months

train = airline.iloc[:TRAIN_SIZE]
test = airline.iloc[TRAIN_SIZE:]

print(f"Training set: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} obs)")
print(f"Test set:     {test.index[0].date()} to {test.index[-1].date()} ({len(test)} obs)")

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Train")
ax.plot(test["Passengers"], label="Test", color="tab:orange")
ax.axvline(test.index[0], color="grey", linestyle="--", alpha=0.7, label="Split point")
ax.set_title("Airline Passengers — Train / Test Split")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

---
## 1. Mean Method

The simplest possible forecast: predict the **historical mean** for all
future time steps.

$$
\hat{y}_{T+h|T} = \bar{y} = \frac{1}{T}\sum_{t=1}^{T} y_t
$$

This works well when the series has no trend and no seasonality — just
random fluctuations around a constant level.

In [ ]:
def forecast_mean(train_series, horizon):
    """Mean method: forecast is the training-set mean for all horizons."""
    mean_val = train_series.mean()
    return np.full(horizon, mean_val)


fc_mean = forecast_mean(train["Passengers"], HORIZON)
print(f"Mean forecast (constant): {fc_mean[0]:.2f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Train")
ax.plot(test["Passengers"], label="Test (actual)", color="tab:orange")
ax.plot(test.index, fc_mean, label="Mean forecast", color="tab:red", linestyle="--")
ax.set_title("Mean Method")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("The mean method produces a flat horizontal line.")
print("It completely ignores the upward trend and seasonal pattern.")

---
## 2. Naive Method

The naive method forecasts **the last observed value** for all future periods:

$$
\hat{y}_{T+h|T} = y_T
$$

This is equivalent to a random walk model.  It is a surprisingly hard
benchmark to beat for many financial and economic time series.

In [ ]:
def forecast_naive(train_series, horizon):
    """Naive method: forecast is the last training observation."""
    last_val = train_series.iloc[-1]
    return np.full(horizon, last_val)


fc_naive = forecast_naive(train["Passengers"], HORIZON)
print(f"Naive forecast (constant): {fc_naive[0]:.2f}")
print(f"Last training observation: {train['Passengers'].iloc[-1]}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Train")
ax.plot(test["Passengers"], label="Test (actual)", color="tab:orange")
ax.plot(test.index, fc_naive, label="Naive forecast", color="tab:green", linestyle="--")
ax.set_title("Naive Method")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("Like the mean method, naive produces a flat line — but at the level")
print("of the last observation rather than the historical average.")

---
## 3. Seasonal Naive Method

For data with a clear seasonal pattern, the seasonal naive method forecasts
the value from the **same season in the last year**:

$$
\hat{y}_{T+h|T} = y_{T+h-m(k+1)}
$$

where $m$ is the seasonal period (12 for monthly data) and $k$ is the integer
part of $(h-1)/m$.  In plain English: to forecast January next year, use
January this year.

In [ ]:
def forecast_seasonal_naive(train_series, horizon, season_length=12):
    """Seasonal naive: forecast is the value from the same season last year."""
    # Get the last full season from training data
    last_season = train_series.values[-season_length:]
    # Tile it to cover the forecast horizon
    n_repeats = int(np.ceil(horizon / season_length))
    forecasts = np.tile(last_season, n_repeats)[:horizon]
    return forecasts


fc_snaive = forecast_seasonal_naive(train["Passengers"], HORIZON)
print("Seasonal naive forecasts (first 12 months):")
print(fc_snaive[:12])
print("\nLast 12 training observations:")
print(train["Passengers"].values[-12:])

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Train")
ax.plot(test["Passengers"], label="Test (actual)", color="tab:orange")
ax.plot(test.index, fc_snaive, label="Seasonal naive", color="tab:purple", linestyle="--")
ax.set_title("Seasonal Naive Method")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("Seasonal naive captures the seasonal pattern but misses the upward trend.")
print("It simply replays the last year's pattern.")

---
## 4. Drift Method

The drift method draws a **straight line from the first training observation
to the last** and extrapolates it forward:

$$
\hat{y}_{T+h|T} = y_T + h \left(\frac{y_T - y_1}{T-1}\right)
$$

This is equivalent to a random walk with drift.  The slope equals the
average change per period over the training set.

In [ ]:
def forecast_drift(train_series, horizon):
    """Drift method: extrapolate the line from first to last observation."""
    y_1 = train_series.iloc[0]
    y_T = train_series.iloc[-1]
    T = len(train_series)
    slope = (y_T - y_1) / (T - 1)
    forecasts = np.array([y_T + h * slope for h in range(1, horizon + 1)])
    return forecasts


fc_drift = forecast_drift(train["Passengers"], HORIZON)

slope = (train["Passengers"].iloc[-1] - train["Passengers"].iloc[0]) / (len(train) - 1)
print(f"First observation: {train['Passengers'].iloc[0]}")
print(f"Last observation:  {train['Passengers'].iloc[-1]}")
print(f"Slope (avg change per month): {slope:.4f}")
print(f"\nDrift forecasts (first 6): {fc_drift[:6].round(2)}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Train")
ax.plot(test["Passengers"], label="Test (actual)", color="tab:orange")
ax.plot(test.index, fc_drift, label="Drift forecast", color="tab:brown", linestyle="--")
ax.set_title("Drift Method")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("The drift method captures the overall upward trend but ignores seasonality.")
print("It produces a straight line continuation of the training data trajectory.")

---
## 5. All Four Methods Compared

Let us plot all four forecasts together against the actuals to see their
relative strengths and weaknesses.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Training data
ax.plot(train["Passengers"], label="Train", color="black", alpha=0.5)

# Actuals
ax.plot(test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)

# Forecasts
ax.plot(test.index, fc_mean, label="Mean", linestyle="--", color="tab:red")
ax.plot(test.index, fc_naive, label="Naive", linestyle="--", color="tab:green")
ax.plot(test.index, fc_snaive, label="Seasonal Naive", linestyle="--", color="tab:purple")
ax.plot(test.index, fc_drift, label="Drift", linestyle="--", color="tab:brown")

ax.axvline(test.index[0], color="grey", linestyle=":", alpha=0.5)
ax.set_title("All Four Simple Forecasting Methods — Airline Passengers")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Quick accuracy comparison using MAE
from sklearn.metrics import mean_absolute_error

actual = test["Passengers"].values

results = pd.DataFrame({
    "Method": ["Mean", "Naive", "Seasonal Naive", "Drift"],
    "MAE": [
        mean_absolute_error(actual, fc_mean),
        mean_absolute_error(actual, fc_naive),
        mean_absolute_error(actual, fc_snaive),
        mean_absolute_error(actual, fc_drift),
    ],
})
results["MAE"] = results["MAE"].round(2)
results = results.sort_values("MAE").reset_index(drop=True)
print(results.to_string(index=False))
print(f"\nBest simple method: {results.iloc[0]['Method']} with MAE = {results.iloc[0]['MAE']}")

---
## 6. Understanding Each Method Geometrically

Each method makes a different assumption about the data-generating process:

- **Mean**: the series is stationary with no trend or seasonality — future
  values hover around a fixed level.
- **Naive**: the series is a random walk — the best guess for tomorrow is
  today's value.
- **Seasonal Naive**: the series has a repeating seasonal pattern — the best
  guess for next January is this January.
- **Drift**: the series follows a linear trend — extrapolate the overall
  direction.

In [ ]:
# Geometric illustration: show each method's "logic" on a simplified series
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, fc, title, color in zip(
    axes.flat,
    [fc_mean, fc_naive, fc_snaive, fc_drift],
    ["Mean Method", "Naive Method", "Seasonal Naive", "Drift Method"],
    ["tab:red", "tab:green", "tab:purple", "tab:brown"],
):
    # Show only last 36 months of training for clarity
    ax.plot(train["Passengers"].iloc[-36:], color="black", alpha=0.5, label="Train (last 3 yr)")
    ax.plot(test["Passengers"], color="tab:blue", linewidth=2, label="Actual")
    ax.plot(test.index, fc, color=color, linestyle="--", linewidth=2, label="Forecast")
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylabel("Passengers")

plt.tight_layout()
plt.show()

---
## 7. Fitted Values (In-Sample One-Step-Ahead Forecasts)

The **fitted value** at time $t$ is the one-step-ahead forecast using
all data up to time $t-1$.  Understanding fitted values is essential
for computing residuals (next notebook).

In [ ]:
def fitted_mean(series):
    """Fitted values for mean method: expanding mean up to t-1."""
    fitted = series.expanding().mean().shift(1)
    return fitted


def fitted_naive(series):
    """Fitted values for naive method: y_{t-1}."""
    return series.shift(1)


def fitted_seasonal_naive(series, season_length=12):
    """Fitted values for seasonal naive: y_{t-m}."""
    return series.shift(season_length)


def fitted_drift(series):
    """Fitted values for drift method."""
    fitted = pd.Series(np.nan, index=series.index)
    for t in range(2, len(series)):
        y_1 = series.iloc[0]
        y_prev = series.iloc[t - 1]
        slope = (y_prev - y_1) / (t - 1)
        fitted.iloc[t] = y_prev + slope
    return fitted


print("Fitted value functions defined.")

In [ ]:
# Compute fitted values for the naive method on training data
fitted_vals = fitted_naive(train["Passengers"])

fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Actual", color="tab:blue")
ax.plot(fitted_vals, label="Fitted (naive)", color="tab:red", alpha=0.7, linestyle="--")
ax.set_title("Naive Method — Fitted Values (one-step-ahead)")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("For the naive method, the fitted value at time t is simply y_{t-1}.")
print("The fitted series is the original shifted by one period.")

In [ ]:
# Compute fitted values for seasonal naive
fitted_sn = fitted_seasonal_naive(train["Passengers"])

fig, ax = plt.subplots()
ax.plot(train["Passengers"], label="Actual", color="tab:blue")
ax.plot(fitted_sn, label="Fitted (seasonal naive)", color="tab:purple", alpha=0.7, linestyle="--")
ax.set_title("Seasonal Naive — Fitted Values")
ax.set_ylabel("Thousands of Passengers")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("For seasonal naive, the fitted value at time t is y_{t-12}.")
print("The first 12 months have no fitted values (no previous year).")

---
## 8. Implementing with statsmodels

While implementing from scratch builds understanding, `statsmodels` provides
ready-made implementations for some of these methods.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Naive method is equivalent to Simple Exponential Smoothing with alpha=1
# But we can also use a more direct approach

# The mean method is equivalent to SES with alpha close to 0
# (or equivalently, ExponentialSmoothing with no trend and no seasonal)

print("statsmodels doesn't have explicit 'naive' or 'mean' functions.")
print("These are typically implemented directly as shown above.")
print("\nHowever, we can verify our implementations match statsmodels results:")

# SES with alpha=1 should match naive
ses_model = ExponentialSmoothing(
    train["Passengers"],
    trend=None,
    seasonal=None,
).fit(smoothing_level=0.9999, optimized=False)

ses_forecast = ses_model.forecast(HORIZON)
print(f"\nSES (alpha~1) forecast: {ses_forecast.iloc[0]:.2f}")
print(f"Our naive forecast:     {fc_naive[0]:.2f}")

In [ ]:
# Seasonal naive using statsmodels
# ExponentialSmoothing with seasonal='add' or 'mul' and very high smoothing parameters
# approximates seasonal naive, but a direct implementation is clearer.

# Let's verify our seasonal naive matches a direct look-up
for h in range(1, 13):
    month_idx = h - 1
    expected = train["Passengers"].iloc[-(12 - month_idx)]
    actual_fc = fc_snaive[month_idx]
    match = "OK" if expected == actual_fc else "MISMATCH"
    # print for first few
    if h <= 6:
        print(f"  h={h}: last year = {expected}, forecast = {actual_fc}  [{match}]")

print("\nAll forecasts correctly reproduce last year's values.")

---
## 9. When to Use Each Method

| Method | Best when... | Typical data |
|---|---|---|
| Mean | No trend, no seasonality, long stable history | Stationary processes |
| Naive | Strong random-walk behaviour, no seasonality | Stock prices, exchange rates |
| Seasonal Naive | Clear seasonal pattern, stable level | Retail sales, temperature |
| Drift | Clear trend, no seasonality | Population growth, GDP |

**Key insight**: these are not "bad" methods — they are the *minimum standard*
against which all other methods are measured.  A model that cannot beat
seasonal naive on seasonal data is adding complexity without adding value.

In [ ]:
# Demonstrate on a second dataset: daily births (no strong seasonality)
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]
print(f"Daily births shape: {births.shape}")
births.head()

In [ ]:
# Split births: first 335 days train, last 30 days test
births_train = births.iloc[:335]
births_test = births.iloc[335:]
births_h = len(births_test)

# Apply all four methods
b_mean = forecast_mean(births_train["Births"], births_h)
b_naive = forecast_naive(births_train["Births"], births_h)
b_snaive = forecast_seasonal_naive(births_train["Births"], births_h, season_length=7)
b_drift = forecast_drift(births_train["Births"], births_h)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(births_train["Births"].iloc[-60:], label="Train (last 60 days)", color="black", alpha=0.5)
ax.plot(births_test["Births"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(births_test.index, b_mean, label="Mean", linestyle="--", color="tab:red")
ax.plot(births_test.index, b_naive, label="Naive", linestyle="--", color="tab:green")
ax.plot(births_test.index, b_snaive, label="Seasonal Naive (m=7)", linestyle="--", color="tab:purple")
ax.plot(births_test.index, b_drift, label="Drift", linestyle="--", color="tab:brown")
ax.set_title("Simple Methods on Daily Births (no strong trend or seasonality)")
ax.set_ylabel("Number of Births")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Compare MAE on births
births_actual = births_test["Births"].values

births_results = pd.DataFrame({
    "Method": ["Mean", "Naive", "Seasonal Naive (m=7)", "Drift"],
    "MAE": [
        mean_absolute_error(births_actual, b_mean),
        mean_absolute_error(births_actual, b_naive),
        mean_absolute_error(births_actual, b_snaive),
        mean_absolute_error(births_actual, b_drift),
    ],
})
births_results["MAE"] = births_results["MAE"].round(2)
births_results = births_results.sort_values("MAE").reset_index(drop=True)
print(births_results.to_string(index=False))
print(f"\nFor data without strong seasonality, the mean method often performs well.")

---
## 10. Combining Benchmarks: Choosing the Right One

A practical approach: **compute all four benchmarks, pick the best one as
your baseline**, and then try to beat it with more sophisticated models.

The choice of benchmark should match the data characteristics:
- Trending data → drift (or naive if trend is stochastic)
- Seasonal data → seasonal naive
- Stationary data → mean
- Random walk data → naive

In [ ]:
def benchmark_summary(train_series, test_series, season_length=12):
    """Compute all four benchmarks and return a results DataFrame."""
    h = len(test_series)
    actual = test_series.values

    methods = {
        "Mean": forecast_mean(train_series, h),
        "Naive": forecast_naive(train_series, h),
        "Seasonal Naive": forecast_seasonal_naive(train_series, h, season_length),
        "Drift": forecast_drift(train_series, h),
    }

    rows = []
    for name, fc in methods.items():
        mae = mean_absolute_error(actual, fc)
        rows.append({"Method": name, "MAE": round(mae, 2)})

    df = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
    return df


print("=== Airline Passengers ===")
print(benchmark_summary(train["Passengers"], test["Passengers"]).to_string(index=False))
print()
print("=== Daily Births ===")
print(benchmark_summary(births_train["Births"], births_test["Births"], season_length=7).to_string(index=False))

---
## Summary

- **Mean, naive, seasonal naive, and drift** are the four foundational
  benchmark methods that every forecaster must know.
- They are **trivially simple**, require **no tuning**, and set the
  **minimum bar** that any more complex model must clear.
- Each makes a different structural assumption (constant level, random walk,
  seasonal repetition, linear trend) — the right benchmark depends on the data.
- **Fitted values** (one-step-ahead in-sample forecasts) are needed to
  compute residuals, which are the subject of the next notebook.
- **Always start with benchmarks** before trying anything more sophisticated.
  If seasonal naive has a MAE of 30 and your neural network has a MAE of 32,
  the neural network is not helping — it is just adding complexity.

In [ ]:
print("Next notebook: Residual Diagnostics — how to check whether a model")
print("has captured all the available signal in the data.")